# P19 — Prédiction des jours de pic de NO2 à Madrid (Comunidad de Madrid)

**Contexte métier.** La Comunidad de Madrid exploite un réseau de stations de mesure horaires. En cas de dépassement des seuils NO2, la mairie doit déclencher des mesures (circulation restreinte, alertes aux personnes sensibles). Anticiper à J+1 permet d'activer les protocoles avant le pic.

**Objectif de l'analyse exploratoire.** Construire une série journalière par station à partir du format horaire « large », puis identifier la cible (jour dégradé) et les régresseurs (lags, saisonnalité, station).

**Données.** datos.comunidad.madrid (CKAN) — « Red de Calidad del Aire, datos horarios 2024 », 65 516 lignes (station x polluant x jour x 24 heures), code de station relatif à la commune.

## 1. Téléchargement

Fichier annuel 2024 du CKAN de la Comunidad de Madrid (format CSV « large » : une ligne par station-polluant-jour, 24 colonnes horaires + 24 drapeaux de validation).

In [ ]:
import os, urllib.request

os.makedirs("data", exist_ok=True)
URL = ("https://datos.comunidad.madrid/dataset/f18aa765-7432-4b39-8b6e-c41799c1c3e0/resource/"
       "519afc1d-55e7-4e9a-91c4-af0d71e1fc14/download/2024.csv")
if not os.path.exists("data/madrid_air_2024.csv"):
    urllib.request.urlretrieve(URL, "data/madrid_air_2024.csv")
print("OK", os.path.getsize("data/madrid_air_2024.csv") // 1024, "Ko")

## 2. Chargement

Séparateur « ; », décimales à virgule. Le code `estacion` est relatif au `municipio` : une station est la paire (municipio, estacion), sinon on fusionne des stations différentes.

In [ ]:
import pandas as pd

df = pd.read_csv("data/madrid_air_2024.csv", sep=";", decimal=",")
print(df.shape, "lignes x colonnes")
print("Stations physiques :", df.groupby(["municipio", "estacion"]).ngroups)
print("Polluants (magnitud) :", sorted(df["magnitud"].unique()))
print(df[["provincia", "municipio", "estacion", "magnitud"]].head(4).to_string())

## 3. Analyse

On dé-pivote les 24 colonnes horaires, on ne garde que les heures validées (drapeau V), puis on agrège en moyenne journalière par station physique. NO2 = magnitud 8 ; jour dégradé = au moins une station > 40 µg/m³.

In [ ]:
horaires = [f"h{i:02d}" for i in range(1, 25)]
valides = [f"v{i:02d}" for i in range(1, 25)]
flags = df[valides].to_numpy().ravel()
print("Heures validées :", round(100 * (flags == "V").mean(), 1), "%")

ids = ["municipio", "estacion", "magnitud", "ano", "mes", "dia"]
long = df.melt(id_vars=ids, value_vars=horaires, var_name="heure", value_name="val")
vflags = df.melt(id_vars=ids, value_vars=valides, var_name="heure_v", value_name="flag")
long["flag"] = vflags["flag"].to_numpy()
long["val"] = pd.to_numeric(long["val"], errors="coerce")
long = long[long["flag"] == "V"]
long = long.rename(columns={"ano": "year", "mes": "month", "dia": "day"})
long["jour"] = pd.to_datetime(long[["year", "month", "day"]])

daily = long.groupby(["municipio", "estacion", "magnitud", "jour"])["val"].mean().reset_index()
no2 = daily[daily["magnitud"] == 8].pivot(index="jour", columns=["municipio", "estacion"], values="val")
print("NO2 :", no2.shape[1], "stations x", no2.shape[0], "jours")

reseau = no2.mean(axis=1)
degrade = (no2 > 40).any(axis=1)
print("Jours avec au moins une station > 40 µg/m³ :", round(100 * degrade.mean(), 1), "%")
print("Moyenne réseau :", round(reseau.mean(), 1), "µg/m³ ; médiane :", round(reseau.median(), 1))
print("Corrélation J / J-1 :", round(reseau.corr(reseau.shift(1)), 2),
      " ; J / J-7 :", round(reseau.corr(reseau.shift(7)), 2))
print()
print("NO2 moyen par mois :")
print(reseau.groupby(reseau.index.month).mean().round(1).to_string())

## 4. Résultats observés (exécution du 16/10/2026)

- fichier brut : 65 516 lignes x 56 colonnes (une ligne = station x polluant x jour, 24 heures + drapeaux)
- 73 stations physiques (paires municipio-estacion), 13 polluants ; 97,5 % des heures validées
- NO2 (magnitud 8) : 28 stations x 366 jours
- jours dégradés (au moins une station > 40 µg/m³) : 29,5 % de l'année 2024
- moyenne réseau : 15,6 µg/m³ ; médiane 13,1 ; 95 % : 31,2 ; max 41,9
- NO2 par mois : janvier 23,4, février 18,8, décembre 26,1 vs juin 10,0, mai 10,3 (hiver très marqué)
- corrélation J / J-1 : 0,79 ; J / J-7 : 0,34
- stations les plus chargées : 49-3 (24,3 µg/m³ médiane), 74-7 (24,1), 65-14 (20,8)

## 5. Variable cible et régresseurs

**Cible (classification binaire) : jour dégradé à J+1** — au moins une station NO2 > 40 µg/m³ le lendemain (29,5 % de positifs sur 2024 : problème équilibré, directement actionnable par la cellule qualité de l'air).

**Régresseurs retenus :**
- lags de la série : NO2 réseau J (corr. 0,79), J-2, J-7, max des stations à J
- station ou ville : médiane de chaque station, station déjà au-dessus du seuil à J
- saisonnalité : mois (décembre 26,1 vs juin 10,0), jour de semaine (trafic)
- polluants corrélés à J : PM10, O3 (magnitud 10 et 14 du même fichier)

**Pièges :**
- le code `estacion` est relatif au `municipio` : ne jamais agréger sur `estacion` seul (7 codes fusionnés au lieu de 28 stations)
- les drapeaux de validation (V/N) : une donnée non validée n'est pas une donnée absente ; le taux de 97,5 % masque des trous de maintenance par station
- format « large » 24 heures + décimales à virgule : lecture délicate (sep=';', decimal=',')
- découpage temporel strict : entraîner sur janvier-octobre, tester sur novembre-décembre (saisonnalité forte)
- 2024 = une seule année : compléter avec 2023 et 2022 (même CKAN) avant de généraliser